In [1]:
import psycopg2
import pandas as pd

def fetch_table_to_dataframe(host_ip, database_name, user, password, table_name, port=5432):
    try:
        connection = psycopg2.connect(
            host=host_ip, database=database_name, user=user, password=password, port=port
        )
        print(f"Connected successfully to {database_name} on {host_ip}")
        df = pd.read_sql_query(f"SELECT * FROM {table_name};", connection)
        print(f"✅ Fetched {len(df)} rows from '{table_name}'")
        return df
    except Exception as e:
        print(f"❌ Error: {e}")
        return None
    finally:
        if 'connection' in locals():
            connection.close()

# --- Configuration ---
HOST_IP = "100.94.14.115"
DATABASE_NAME = "pradigma-extractor"
USER = "postgres"
PASSWORD = "password"
PORT = 5432
TABLE_NAME = "extraction"

df_original = fetch_table_to_dataframe(HOST_IP, DATABASE_NAME, USER, PASSWORD, TABLE_NAME, PORT)

Connected successfully to pradigma-extractor on 100.94.14.115


C:\Users\win 11\AppData\Local\Temp\ipykernel_18708\1965381518.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(f"SELECT * FROM {table_name};", connection)


✅ Fetched 7990 rows from 'extraction'


In [10]:
df = df_original.copy(deep=True)
df = df[
    (df['dept_name'] == 'Signalling-And-Communication') & (df['status_id'] == 1)
][['filename', 'workorder_id', 'interval', 'dept_name', 'json_data']]

df

,filename,workorder_id,interval,dept_name,json_data
2,SC_PM_NA_CCTV_NA_19.pdf,NaN,Unknown,Signalling-And-Communication,"{'notification': {'notification_no': 'NA', 'no..."
219,SC_PM_NA_CCTV_NA_20 (2).pdf,NaN,Unknown,Signalling-And-Communication,"{'notification': {'notification_no': 'NA', 'no..."
569,SC_PM_QTR_UPS_4000608017.pdf,4.000608e+09,Quarterly,Signalling-And-Communication,{'notification': {'notification_no': '12290095...
740,SC_PM_NA_CCTV_NA_2 (2).pdf,NaN,Unknown,Signalling-And-Communication,"{'notification': {'notification_no': 'NA', 'no..."
781,SC_PM_NA_CCTV_NA_23 (1).pdf,NaN,Unknown,Signalling-And-Communication,"{'notification': {'notification_no': 'NA', 'no..."
...,...,...,...,...,...
7985,SC_PM_NA_WiFi_NA_104.pdf,NaN,Unknown,Signalling-And-Communication,"{'notification': {'notification_no': 'NA', 'no..."
7986,SC_PM_NA_WiFi_NA_102.pdf,NaN,Unknown,Signalling-And-Communication,"{'notification': {'notification_no': 'NA', 'no..."
7987,SC_PM_NA_WiFi_NA_100.pdf,NaN,Unknown,Signalling-And-Communication,"{'notification': {'notification_no': 'NA', 'no..."
7988,SC_PM_NA_WiFi_NA.pdf,NaN,Unknown,Signalling-And-Communication,"{'notification': {'notification_no': 'NA', 'no..."


In [11]:
import json

def normalize(x):
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        try:
            return json.loads(x)
        except:
            return None
    return None

df["json_data"] = df["json_data"].apply(normalize)

In [12]:
def safe_get(d, keys):
    for k in keys:
        if not isinstance(d, dict):
            return None
        d = d.get(k)
    return d

In [13]:
def find_key(data, target_key):
    if isinstance(data, dict):
        for k, v in data.items():
            if k == target_key:
                return v
            result = find_key(v, target_key)
            if result is not None:
                return result

    elif isinstance(data, list):
        for item in data:
            result = find_key(item, target_key)
            if result is not None:
                return result

    return None

extracted_df = pd.DataFrame({

    "workorder_no": df["workorder_id"],
    
    "inspection_date": df["json_data"].apply(lambda x: find_key(x, "datetime")),
    
    "supervisor_id": df["json_data"].apply(lambda x: find_key(x, "performed_by")),

    "technician_ids": df["json_data"].apply(lambda x: find_key(x, "verified_by")),
    
    "filename": df["filename"],

})

extracted_df.head()

,workorder_no,inspection_date,supervisor_id,technician_ids,filename
2,NaN,22/02/2022,7223,7111,SC_PM_NA_CCTV_NA_19.pdf
219,NaN,16/09/2023,11727,7111,SC_PM_NA_CCTV_NA_20 (2).pdf
569,4.000608e+09,None,19921,7111,SC_PM_QTR_UPS_4000608017.pdf
740,NaN,22/05/2023,11727,7111,SC_PM_NA_CCTV_NA_2 (2).pdf
781,NaN,14/06/2023,7222,7111,SC_PM_NA_CCTV_NA_23 (1).pdf


In [14]:
extracted_df.to_excel("extracted/extracted_snc.xlsx", index=False)

### Comparing list of technician and supervisor with tbl_user

In [15]:
ref_user = pd.read_excel("tbl_users.xlsx")
ref_user.head()

,id,staff_id,name,call_sign,position,department,email
0,1,10018972,LUQMAN NULHAKIM BIN JAMALUDIN,LNJ 18972,Senior Associate,Power System,luqnman.jamaludin@prasarana.com.my
1,2,10007223,ROHAIZAN BIN MASTOR,RM7223,Associate,Power System,rohaizan@prasarana.com.my
2,3,10023969,MUHAMMAD NUR ZAM ZAM BIN MOHD ROSLE,NZZ23969,Associate,Power System,nurzamzam.rosle@prasarana.com.my
3,4,10025298,IDHAR DANIEL BIN MOHD AZHAR,IDA25298,Associate,Power System,idhar.azhar@prasarana.com.my
4,5,10024324,MUHAMMAD IKHWAN BIN ABDULLAH,MIA24324,Senior Associate,Power System,ikhwan.abdullah@prasarana.com.my


In [16]:
cleaned_user = ref_user[["staff_id", "name", "call_sign", "department"]].copy()

cleaned_user["stamp_id"] = (
    cleaned_user["staff_id"]
    .astype(str)
    .str.replace(r"^1000|^100", "", regex=True)
)

cleaned_user["stamp_id"] = cleaned_user["stamp_id"].astype(str)
cleaned_user.head()

,staff_id,name,call_sign,department,stamp_id
0,10018972,LUQMAN NULHAKIM BIN JAMALUDIN,LNJ 18972,Power System,18972
1,10007223,ROHAIZAN BIN MASTOR,RM7223,Power System,7223
2,10023969,MUHAMMAD NUR ZAM ZAM BIN MOHD ROSLE,NZZ23969,Power System,23969
3,10025298,IDHAR DANIEL BIN MOHD AZHAR,IDA25298,Power System,25298
4,10024324,MUHAMMAD IKHWAN BIN ABDULLAH,MIA24324,Power System,24324


In [18]:
import pandas as pd

df_snc = pd.read_excel("extracted/extracted_snc.xlsx")
df_snc.head()

,workorder_no,inspection_date,supervisor_id,technician_ids,filename
0,NaN,22/02/2022,7223,7111,SC_PM_NA_CCTV_NA_19.pdf
1,NaN,16/09/2023,11727,7111,SC_PM_NA_CCTV_NA_20 (2).pdf
2,4.000608e+09,NaN,19921,7111,SC_PM_QTR_UPS_4000608017.pdf
3,NaN,22/05/2023,11727,7111,SC_PM_NA_CCTV_NA_2 (2).pdf
4,NaN,14/06/2023,7222,7111,SC_PM_NA_CCTV_NA_23 (1).pdf


In [19]:
id_to_name = dict(zip(
    cleaned_user["stamp_id"].astype(str),
    cleaned_user["name"]
))

id_to_supervisor_name = id_to_name

def build_rows(row):
    
    tech_ids = row["technician_ids"]

    if not isinstance(tech_ids, str):
        return []

    tech_list = [i.strip() for i in tech_ids.split(",") if i.strip()]

    results = []

    for tech_id in tech_list:
        
        results.append({
            "filename": row.get("filename"),
            "workorder_no": row.get("workorder_no"),
            "inspection_date": row.get("inspection_date"),

            "technician_id": tech_id,
            "name": id_to_name.get(tech_id, ""),

            "supervisor_id": row.get("supervisor_id"),
            "supervisor_name": id_to_supervisor_name.get(
                str(row.get("supervisor_id")), ""
            ),
        })

    return results

expanded = df_snc.apply(build_rows, axis=1).explode().dropna()

final_df = pd.DataFrame(expanded.tolist())
final_df.head()

,filename,workorder_no,inspection_date,technician_id,name,supervisor_id,supervisor_name
0,SC_PM_NA_CCTV_NA_19.pdf,NaN,22/02/2022,7111,AZHAR BIN YUNUS,7223,ROHAIZAN BIN MASTOR
1,SC_PM_NA_CCTV_NA_2 (2).pdf,NaN,22/05/2023,7111,AZHAR BIN YUNUS,11727,ABDUL AZIM BIN NASIR
2,SC_PM_NA_CCTV_NA_23 (1).pdf,NaN,14/06/2023,7111,AZHAR BIN YUNUS,7222,MOHD RAZIF BIN RAZALI
3,SC_PM_QTR_CCTV_4000480655.pdf,4.000481e+09,21/08/2022,7111,AZHAR BIN YUNUS,11727,ABDUL AZIM BIN NASIR
4,SC_PM_QTR_Signalling_4000541173.pdf,4.000541e+09,NaN,16079,MUHAMMAD TAQIUDDIN SHAH BIN SHASHIM SHAH,7222,MOHD RAZIF BIN RAZALI


In [20]:
final_df.to_excel("output/staff_snc.xlsx", index=False)